# Data QA

End-to-end checks of the data plan (section 3). Run `uv run python -m statarb` first; every list is also in `data/qa.sqlite`.

In [1]:
import pandas as pd

from statarb import qa
from statarb.io import load

t = qa.run()
t["summary"]

,check,value,expected
0,coverage of index stock-days,99.974%,> 99%
1,names per day (min / median),488 / 503,"~500, min >= 480"
2,unexplained |r| > 40%,145,short list
3,split days with |r| > 40%,0,0
4,stale / no-trade index days,217,rare
5,tickers still disagreeing across sources,0,"few, listed"
6,Tiingo vs WIKI days over 1%,770,listed
7,SPY vs members corr (min year),0.825,"high, stable"


## Coverage and universe

In [2]:
names = t["names_per_day"]["names"]
assert len(t["missing"]) / (len(t["missing"]) + len(load("returns").query("in_index"))) < 1e-3
assert names.min() >= qa.MIN_NAMES and 495 <= names.median() <= 510
r = load("returns")
assert not r.duplicated(["date", "ticker"]).any()
names.describe()

count    5030.000000
mean      502.428429
std         2.397874
min       488.000000
25%       500.000000
50%       503.000000
75%       505.000000
max       507.000000
Name: names, dtype: float64

## Returns: splits, jumps, delistings

In [3]:
def ret(ticker, day):
    return r.loc[(r.ticker == ticker) & (r.date == day), "ret"].item()


assert (t["split_days"]["ret"].abs() > qa.JUMP).sum() == 0
assert abs(ret("AAPL", "2020-08-31") - 0.0339) < 1e-3  # 4:1 split day
assert abs(ret("TSLA", "2020-08-31") - 0.1257) < 1e-3  # 5:1 split day
assert abs(ret("SIVB", "2023-03-10") + 0.996228) < 1e-6  # hand-listed delisting
assert len(t["jumps"]) < 200
t["jumps"].sort_values("ret").head(10)

,date,ticker,ret,close,source
90,2008-09-15,LEHMQ,-0.942466,0.2100,sp500data:crsp
133,2008-09-26,WAMUQ,-0.905089,0.1604,sp500data:crsp
59,2023-05-03,FRC,-0.904957,0.3336,tiingo
54,2008-09-08,FNMA,-0.896307,0.7300,tiingo
20,2008-03-17,BSC,-0.839667,4.8100,sp500data:crsp
53,2008-09-08,FMCC,-0.827451,0.8800,tiingo
135,2008-09-29,WB,-0.816000,1.8400,sp500data:crsp
30,2009-07-16,CITGQ,-0.750000,0.4100,sp500data:crsp
65,2008-11-11,GGP,-0.643066,0.4890,tiingo
102,2008-09-29,NCC,-0.633423,1.3600,sp500data:crsp


## Cross-source agreement and market check

In [4]:
assert t["suspects"].empty
spy = t["spy_vs_members"]
assert spy["corr"].min() > 0.8 and spy["corr"].median() > 0.95  # 2024 is lowest: mega-cap concentration
spy.round(3).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
year,2006.000,2007.000,2008.00,2009.000,2010.000,2011.000,2012.000,2013.000,2014.000,2015.000,2016.000,2017.00,2018.000,2019.000,2020.000,2021.000,2022.000,2023.000,2024.000,2025.000
corr,0.945,0.973,0.97,0.986,0.991,0.994,0.982,0.982,0.985,0.984,0.971,0.93,0.982,0.969,0.954,0.897,0.977,0.906,0.825,0.907


## Sector and spread

In [5]:
sec = load("sector").set_index(["date", "ticker"])["sector"]
day = pd.Timestamp("2015-06-01")
assert sec[(day, "XOM")] == "XLE" and sec[(day, "JPM")] == "XLF" and sec[(day, "DUK")] == "XLU"
sp = load("spread")
ar = sp.loc[sp.spread_source == "ar", "spread"]
assert (sp.spread_source == "ar").mean() > 0.95 and 0.0005 < ar.median() < 0.006
(ar.describe() * 1e4).round(1)

count    2.441738e+10
mean     4.640000e+01
std      6.840000e+01
min      0.000000e+00
25%      0.000000e+00
50%      2.350000e+01
75%      7.280000e+01
max      2.242100e+03
Name: spread, dtype: float64